1. Data Understanding

In [1]:
#install library kaggle untuk mengakses dataset dari Kaggle
!pip install kaggle

In [2]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mahdimashayekhi/disease-risk-from-daily-habits")
print("Path to dataset files:", path)

100%|██████████| 20.8M/20.8M [00:02<00:00, 8.03MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mahdimashayekhi/disease-risk-from-daily-habits/versions/1


In [3]:
#install library catboost untuk model machine learning
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:000:00:0100:01


In [4]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV

In [5]:
# mengecek file yang telah di download
print(os.listdir(path))

['health_lifestyle_classification.csv']


In [6]:
# Membaca dataset
df = pd.read_csv(path + "/health_lifestyle_classification.csv")
df.head()

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,...,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target
0,1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,...,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502,healthy
1,2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,...,High,5,High,Yes,No,0,1.0,5.5,6.239340,healthy
2,3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,...,High,4,Moderate,No,No,0,1.0,5.5,5.423737,healthy
3,4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,...,High,1,NaN,No,Yes,0,1.0,5.5,8.388611,healthy
4,5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,...,High,1,High,Yes,Yes,0,1.0,5.5,0.332622,healthy


In [7]:
# mengecek jumlah baris dan kolom pada dataset
print("data shape : ", df.shape)

data shape :  (100000, 48)


In [8]:
# Mengecek informasi dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 48 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   survey_code               100000 non-null  int64  
 1   age                       100000 non-null  int64  
 2   gender                    100000 non-null  object 
 3   height                    100000 non-null  float64
 4   weight                    100000 non-null  float64
 5   bmi                       100000 non-null  float64
 6   bmi_estimated             100000 non-null  float64
 7   bmi_scaled                100000 non-null  float64
 8   bmi_corrected             100000 non-null  float64
 9   waist_size                100000 non-null  float64
 10  blood_pressure            92331 non-null   float64
 11  heart_rate                85997 non-null   float64
 12  cholesterol               100000 non-null  float64
 13  glucose                   100000 non-null  fl

In [9]:
# describe dataset untuk melihat statistik dasar dari dataset
df.describe()

,survey_code,age,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,...,water_intake,screen_time,stress_level,mental_health_score,income,meals_per_day,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,92331.000000,...,100000.000000,100000.000000,100000.000000,100000.000000,91530.000000,100000.000000,100000.0,89526.0,100000.0,100000.000000
mean,50000.500000,48.525990,170.023707,70.064862,24.493876,24.493876,73.481627,24.494140,84.933043,119.980149,...,2.006373,6.021525,4.991600,5.004680,4038.127284,2.998720,0.0,1.0,5.5,0.015726
std,28867.657797,17.886768,9.982798,14.693667,5.951069,5.951069,17.853206,5.954184,12.040314,15.015503,...,0.688868,2.933835,3.154997,3.164228,1930.025678,1.414786,0.0,0.0,0.0,5.764489
min,1.000000,18.000000,140.000000,40.000000,9.988495,9.988495,29.965484,9.893845,34.093185,59.128168,...,0.500000,0.000000,0.000000,0.000000,500.000000,1.000000,0.0,1.0,5.5,-9.999895
25%,25000.750000,33.000000,163.306615,59.856938,20.271405,20.271405,60.814215,20.271059,76.795185,109.812060,...,1.532011,3.971318,2.000000,2.000000,2665.402843,2.000000,0.0,1.0,5.5,-4.980501
50%,50000.500000,48.000000,170.016778,69.924141,24.156734,24.156734,72.470201,24.151699,84.957139,119.951794,...,2.000659,5.991171,5.000000,5.000000,4004.601345,3.000000,0.0,1.0,5.5,0.015589
75%,75000.250000,64.000000,176.728920,80.027418,28.258696,28.258696,84.776088,28.247648,93.018713,130.120621,...,2.473047,8.024470,8.000000,8.000000,5360.012694,4.000000,0.0,1.0,5.5,5.008424
max,100000.000000,79.000000,210.000000,139.250894,59.234792,59.234792,177.704377,59.142646,133.153631,184.439195,...,5.000000,16.000000,10.000000,10.000000,12029.409353,5.000000,0.0,1.0,5.5,9.999966


2. Cleaning Data

# missing value

In [10]:
# Mengecek nilai yang hilang
missing_values = df.isnull().sum()

print("missing value per kolom : ")
print(missing_values[missing_values>0])

missing value per kolom : 
blood_pressure          7669
heart_rate             14003
insulin                15836
daily_steps             8329
alcohol_consumption    42387
income                  8470
exercise_type          24969
caffeine_intake        33261
gene_marker_flag       10474
dtype: int64


In [11]:
# Mengecek persentase nilai yang hilang
missing_percentage = df.isnull().mean().sort_values(ascending=False)*100
print("percentage of missing values per column : ")
print(missing_percentage[missing_percentage>0])

percentage of missing values per column : 
alcohol_consumption    42.387
caffeine_intake        33.261
exercise_type          24.969
insulin                15.836
heart_rate             14.003
gene_marker_flag       10.474
income                  8.470
daily_steps             8.329
blood_pressure          7.669
dtype: float64


In [12]:
# Mengecek jumlah nilai yang hilang per kolom
print("Jumlah Missing Value tiap kolom:\n")

print(df.isna().sum())

Jumlah Missing Value tiap kolom:

survey_code                     0
age                             0
gender                          0
height                          0
weight                          0
bmi                             0
bmi_estimated                   0
bmi_scaled                      0
bmi_corrected                   0
waist_size                      0
blood_pressure               7669
heart_rate                  14003
cholesterol                     0
glucose                         0
insulin                     15836
sleep_hours                     0
sleep_quality                   0
work_hours                      0
physical_activity               0
daily_steps                  8329
calorie_intake                  0
sugar_intake                    0
alcohol_consumption         42387
smoking_level                   0
water_intake                    0
screen_time                     0
stress_level                    0
mental_health_score             0
mental_health_

In [13]:
# Mengecek jumlah nilai yang hilang secara keseluruhan dataset
df.isna().sum().sum()

np.int64(165398)

In [14]:
# Mengisi nilai yang hilang dengan median untuk kolom numerik dan modus untuk kolom kategorikal

# 1) Kolom numerik
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

# 2) Kolom kategorikal
cat_cols = df.select_dtypes(
    include='object'
).columns

for col in cat_cols:
    df[col] = df[col].fillna(
        "Unknown"
    )

In [15]:
(df.isna().sum()/len(df))*100

,0
survey_code,0.0
age,0.0
gender,0.0
height,0.0
weight,0.0
bmi,0.0
bmi_estimated,0.0
bmi_scaled,0.0
bmi_corrected,0.0
waist_size,0.0


In [16]:
df.isna().sum().sum()

np.int64(0)

In [17]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 48 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   survey_code               100000 non-null  int64  
 1   age                       100000 non-null  int64  
 2   gender                    100000 non-null  object 
 3   height                    100000 non-null  float64
 4   weight                    100000 non-null  float64
 5   bmi                       100000 non-null  float64
 6   bmi_estimated             100000 non-null  float64
 7   bmi_scaled                100000 non-null  float64
 8   bmi_corrected             100000 non-null  float64
 9   waist_size                100000 non-null  float64
 10  blood_pressure            100000 non-null  float64
 11  heart_rate                100000 non-null  float64
 12  cholesterol               100000 non-null  float64
 13  glucose                   100000 non-null  fl

# cek duplikat

In [18]:
# Mengecek duplikasi data
df.duplicated().sum()

np.int64(0)

In [19]:
df[df.duplicated()]

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,...,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target


In [20]:
df[df.duplicated(keep=False)]

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,...,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target


In [21]:
# ============================================================
# CEK DUPLICATE DATA
# ============================================================

# Total duplicate
total_duplicate = df.duplicated().sum()

print("Total Duplicate:", total_duplicate)

# Persentase duplicate
duplicate_percent = (
    total_duplicate / len(df)
) * 100

print(
    "Persentase Duplicate:",
    round(duplicate_percent, 2),
    "%"
)

# ============================================================
# TAMPILKAN DATA DUPLICATE
# ============================================================

duplicate_rows = df[
    df.duplicated(keep=False)
]

print("\nData Duplicate:\n")

print(duplicate_rows)

# ============================================================
# JUMLAH ROW DUPLICATE
# ============================================================

print(
    "\nJumlah Row Duplicate:",
    duplicate_rows.shape[0]
)

Total Duplicate: 0
Persentase Duplicate: 0.0 %

Data Duplicate:

Empty DataFrame
Columns: [survey_code, age, gender, height, weight, bmi, bmi_estimated, bmi_scaled, bmi_corrected, waist_size, blood_pressure, heart_rate, cholesterol, glucose, insulin, sleep_hours, sleep_quality, work_hours, physical_activity, daily_steps, calorie_intake, sugar_intake, alcohol_consumption, smoking_level, water_intake, screen_time, stress_level, mental_health_score, mental_health_support, education_level, job_type, occupation, income, diet_type, exercise_type, device_usage, healthcare_access, insurance, sunlight_exposure, meals_per_day, caffeine_intake, family_history, pet_owner, electrolyte_level, gene_marker_flag, environmental_risk_score, daily_supplement_dosage, target]
Index: []

[0 rows x 48 columns]

Jumlah Row Duplicate: 0


In [22]:
df.head()

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,...,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target
0,1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,...,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502,healthy
1,2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,...,High,5,High,Yes,No,0,1.0,5.5,6.239340,healthy
2,3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,...,High,4,Moderate,No,No,0,1.0,5.5,5.423737,healthy
3,4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,...,High,1,Unknown,No,Yes,0,1.0,5.5,8.388611,healthy
4,5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,...,High,1,High,Yes,Yes,0,1.0,5.5,0.332622,healthy


3. Cek Tipe Data

In [23]:
# Mengecek tipe data setiap kolom
print(df.dtypes)

survey_code                   int64
age                           int64
gender                       object
height                      float64
weight                      float64
bmi                         float64
bmi_estimated               float64
bmi_scaled                  float64
bmi_corrected               float64
waist_size                  float64
blood_pressure              float64
heart_rate                  float64
cholesterol                 float64
glucose                     float64
insulin                     float64
sleep_hours                 float64
sleep_quality                object
work_hours                  float64
physical_activity           float64
daily_steps                 float64
calorie_intake              float64
sugar_intake                float64
alcohol_consumption          object
smoking_level                object
water_intake                float64
screen_time                 float64
stress_level                  int64
mental_health_score         

In [24]:
# Mengecek nilai unik pada setiap kolom kategorikal
for col in df.select_dtypes(include='object'):

    print(f"\n===== {col} =====")

    print(df[col].unique())


===== gender =====
['Male' 'Female']

===== sleep_quality =====
['Fair' 'Good' 'Poor' 'Excellent']

===== alcohol_consumption =====
['Unknown' 'Regularly' 'Occasionally']

===== smoking_level =====
['Non-smoker' 'Light' 'Heavy']

===== mental_health_support =====
['No' 'Yes']

===== education_level =====
['PhD' 'High School' 'Master' 'Bachelor']

===== job_type =====
['Tech' 'Office' 'Labor' 'Unemployed' 'Service' 'Healthcare']

===== occupation =====
['Farmer' 'Engineer' 'Teacher' 'Doctor' 'Driver' 'Artist']

===== diet_type =====
['Vegan' 'Vegetarian' 'Omnivore' 'Keto']

===== exercise_type =====
['Strength' 'Cardio' 'Mixed' 'Unknown']

===== device_usage =====
['High' 'Moderate' 'Low']

===== healthcare_access =====
['Poor' 'Moderate' 'Good']

===== insurance =====
['No' 'Yes']

===== sunlight_exposure =====
['High' 'Low' 'Moderate']

===== caffeine_intake =====
['Moderate' 'High' 'Unknown']

===== family_history =====
['No' 'Yes']

===== pet_owner =====
['Yes' 'No']

===== target 

In [25]:
# mengecek semua isi tabel untuk memastikan tidak ada nilai yang aneh atau tidak konsisten

# Tampilkan semua kolom
pd.set_option('display.max_columns', None)

# Lebar tampilan terminal
pd.set_option('display.width', 1000)

# Panjang isi kolom tidak dipotong
pd.set_option('display.max_colwidth', None)

# Tampilkan semua baris jika diperlukan
pd.set_option('display.max_rows', None)

In [26]:
df.head()

,survey_code,age,gender,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,heart_rate,cholesterol,glucose,insulin,sleep_hours,sleep_quality,work_hours,physical_activity,daily_steps,calorie_intake,sugar_intake,alcohol_consumption,smoking_level,water_intake,screen_time,stress_level,mental_health_score,mental_health_support,education_level,job_type,occupation,income,diet_type,exercise_type,device_usage,healthcare_access,insurance,sunlight_exposure,meals_per_day,caffeine_intake,family_history,pet_owner,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage,target
0,1,56,Male,173.416872,56.886640,18.915925,18.915925,56.747776,18.989117,72.165130,118.264254,60.749825,214.580523,103.008176,14.983414,6.475885,Fair,7.671313,0.356918,13320.942595,2673.546960,44.476887,Unknown,Non-smoker,1.694262,5.003963,2,8,No,PhD,Tech,Farmer,6759.821719,Vegan,Strength,High,Poor,No,High,5,Moderate,No,Yes,0,1.0,5.5,-2.275502,healthy
1,2,69,Female,163.207380,97.799859,36.716278,36.716278,110.148833,36.511417,85.598889,117.917986,66.463696,115.794002,116.905134,10.131597,8.428410,Good,9.515198,0.568219,11911.201401,2650.376972,74.663405,Regularly,Light,0.716409,5.925455,3,9,No,High School,Office,Engineer,6240.517690,Vegan,Cardio,Moderate,Moderate,No,High,5,High,Yes,No,0,1.0,5.5,6.239340,healthy
2,3,46,Male,177.281966,80.687562,25.673050,25.673050,77.019151,25.587429,90.295030,123.073698,76.043212,138.134787,89.180302,14.983414,5.702164,Poor,5.829853,3.764406,2974.035375,1746.755144,19.702382,Regularly,Heavy,2.487900,4.371250,0,1,No,Master,Office,Teacher,3429.179266,Vegan,Cardio,High,Good,Yes,High,4,Moderate,No,No,0,1.0,5.5,5.423737,healthy
3,4,32,Female,172.101255,63.142868,21.318480,21.318480,63.955440,21.177109,100.504211,148.173453,68.781981,203.017447,128.375798,18.733179,5.188316,Good,9.489693,0.889474,5321.539497,2034.193242,82.580050,Occasionally,Heavy,2.643335,4.116064,10,4,No,Master,Labor,Teacher,2618.503534,Vegetarian,Mixed,Low,Moderate,No,High,1,Unknown,No,Yes,0,1.0,5.5,8.388611,healthy
4,5,60,Female,163.608816,40.000000,14.943302,14.943302,44.829907,14.844299,69.021150,150.613181,92.335358,200.412439,94.813332,16.038701,7.912514,Good,7.275450,2.901608,9791.376712,2386.210257,45.961322,Unknown,Heavy,1.968393,3.180087,9,7,Yes,Master,Unemployed,Doctor,3662.086276,Vegan,Unknown,Low,Moderate,Yes,High,1,High,Yes,Yes,0,1.0,5.5,0.332622,healthy


In [27]:
cat_cols = [
    'gender',
    'diet_type',
    'exercise_type',
    'smoking_level'
]

for col in cat_cols:
    df[col] = df[col].astype(str)



for col in cat_cols:
    print(col)
    print(df[col].unique())
    print("="*50)

gender
['Male' 'Female']
diet_type
['Vegan' 'Vegetarian' 'Omnivore' 'Keto']
exercise_type
['Strength' 'Cardio' 'Mixed' 'Unknown']
smoking_level
['Non-smoker' 'Light' 'Heavy']


In [28]:
df.describe()

,survey_code,age,height,weight,bmi,bmi_estimated,bmi_scaled,bmi_corrected,waist_size,blood_pressure,heart_rate,cholesterol,glucose,insulin,sleep_hours,work_hours,physical_activity,daily_steps,calorie_intake,sugar_intake,water_intake,screen_time,stress_level,mental_health_score,income,meals_per_day,electrolyte_level,gene_marker_flag,environmental_risk_score,daily_supplement_dosage
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.0,100000.0,100000.0,100000.000000
mean,50000.500000,48.525990,170.023707,70.064862,24.493876,24.493876,73.481627,24.494140,84.933043,119.977974,74.979964,189.966438,99.994538,14.987491,7.002008,8.001331,3.038344,7012.206098,2201.428579,60.047165,2.006373,6.021525,4.991600,5.004680,4035.287636,2.998720,0.0,1.0,5.5,0.015726
std,28867.657797,17.886768,9.982798,14.693667,5.951069,5.951069,17.853206,5.954184,12.040314,14.428246,9.219394,29.981934,19.982828,4.589596,1.496821,1.994723,1.884475,2383.082319,400.516318,19.966850,0.688868,2.933835,3.154997,3.164228,1846.503636,1.414786,0.0,0.0,0.0,5.764489
min,1.000000,18.000000,140.000000,40.000000,9.988495,9.988495,29.965484,9.893845,34.093185,59.128168,34.745092,58.410902,12.434931,-6.794483,3.000000,0.000000,0.000000,1000.000000,527.172360,-27.882444,0.500000,0.000000,0.000000,0.000000,500.000000,1.000000,0.0,1.0,5.5,-9.999895
25%,25000.750000,33.000000,163.306615,59.856938,20.271405,20.271405,60.814215,20.271059,76.795185,110.815119,69.543894,169.667738,86.461401,12.324714,5.986781,6.651093,1.633799,5500.327842,1932.278165,46.504292,1.532011,3.971318,2.000000,2.000000,2796.061969,2.000000,0.0,1.0,5.5,-4.980501
50%,50000.500000,48.000000,170.016778,69.924141,24.156734,24.156734,72.470201,24.151699,84.957139,119.951794,75.046211,190.044579,99.986834,14.983414,6.998164,8.004549,2.971222,7004.285450,2200.992765,60.047905,2.000659,5.991171,5.000000,5.000000,4004.601345,3.000000,0.0,1.0,5.5,0.015589
75%,75000.250000,64.000000,176.728920,80.027418,28.258696,28.258696,84.776088,28.247648,93.018713,129.185145,80.467128,210.222005,113.508722,17.663430,8.019219,9.353669,4.326500,8523.819577,2471.218160,73.476386,2.473047,8.024470,8.000000,8.000000,5219.230324,4.000000,0.0,1.0,5.5,5.008424
max,100000.000000,79.000000,210.000000,139.250894,59.234792,59.234792,177.704377,59.142646,133.153631,184.439195,114.136041,319.875613,183.883548,35.464749,12.000000,16.000000,11.631898,18064.969543,3949.019017,141.514522,5.000000,16.000000,10.000000,10.000000,12029.409353,5.000000,0.0,1.0,5.5,9.999966


In [29]:
print(df['target'].value_counts())
 
print(df['target'].value_counts(normalize=True))

target
healthy     70097
diseased    29903
Name: count, dtype: int64
target
healthy     0.70097
diseased    0.29903
Name: proportion, dtype: float64


In [30]:
# Menghapus kolom yang tidak diperlukan 
df = df.drop(columns=[
'survey_code',
'bmi_estimated','bmi_scaled','bmi_corrected',
'mental_health_score','mental_health_support',
'education_level','job_type','occupation',
'income','healthcare_access','insurance',
'sunlight_exposure','family_history','pet_owner',
'electrolyte_level','gene_marker_flag',
'environmental_risk_score','daily_supplement_dosage'
])

In [31]:
# Mengubah tipe data category menjadi string 
category_cols = df.select_dtypes(include='category').columns

for col in category_cols:
    df[col] = df[col].astype(str)

In [32]:
# Encode target
df['target'] = df['target'].map({
    'healthy': 0,
    'diseased': 1
})

Encoding Kategorikal

In [33]:
X = df.drop('target', axis=1)
y = df['target']

# Ambil semua kolom kategorikal (object)
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()


In [34]:
df.head()   

,age,gender,height,weight,bmi,waist_size,blood_pressure,heart_rate,cholesterol,glucose,insulin,sleep_hours,sleep_quality,work_hours,physical_activity,daily_steps,calorie_intake,sugar_intake,alcohol_consumption,smoking_level,water_intake,screen_time,stress_level,diet_type,exercise_type,device_usage,meals_per_day,caffeine_intake,target
0,56,Male,173.416872,56.886640,18.915925,72.165130,118.264254,60.749825,214.580523,103.008176,14.983414,6.475885,Fair,7.671313,0.356918,13320.942595,2673.546960,44.476887,Unknown,Non-smoker,1.694262,5.003963,2,Vegan,Strength,High,5,Moderate,0
1,69,Female,163.207380,97.799859,36.716278,85.598889,117.917986,66.463696,115.794002,116.905134,10.131597,8.428410,Good,9.515198,0.568219,11911.201401,2650.376972,74.663405,Regularly,Light,0.716409,5.925455,3,Vegan,Cardio,Moderate,5,High,0
2,46,Male,177.281966,80.687562,25.673050,90.295030,123.073698,76.043212,138.134787,89.180302,14.983414,5.702164,Poor,5.829853,3.764406,2974.035375,1746.755144,19.702382,Regularly,Heavy,2.487900,4.371250,0,Vegan,Cardio,High,4,Moderate,0
3,32,Female,172.101255,63.142868,21.318480,100.504211,148.173453,68.781981,203.017447,128.375798,18.733179,5.188316,Good,9.489693,0.889474,5321.539497,2034.193242,82.580050,Occasionally,Heavy,2.643335,4.116064,10,Vegetarian,Mixed,Low,1,Unknown,0
4,60,Female,163.608816,40.000000,14.943302,69.021150,150.613181,92.335358,200.412439,94.813332,16.038701,7.912514,Good,7.275450,2.901608,9791.376712,2386.210257,45.961322,Unknown,Heavy,1.968393,3.180087,9,Vegan,Unknown,Low,1,High,0


In [35]:
df['target'].value_counts()

,count
target,
0,70097
1,29903


In [36]:
# ============================================================
# SPLIT DATA
# ============================================================

from sklearn.model_selection import train_test_split

X = df.drop('target', axis=1)
y = df['target']

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
# Mengecek data sebelum split
print(x_train.shape)

(80000, 28)


Mengatasi Imbalanced Data

In [38]:
# ============================================================
# INSTALL
# ============================================================

!pip install imbalanced-learn

# ============================================================
# IMPORT
# ============================================================

from imblearn.over_sampling import RandomOverSampler

# ============================================================
# CEK DISTRIBUSI SEBELUM
# ============================================================

print("Distribusi target sebelum Oversampling:\n")

print(y_train.value_counts())

# ============================================================
# OVERSAMPLING
# ============================================================

oversample = RandomOverSampler(
    sampling_strategy='auto',
    random_state=42
)

x_train_over, y_train_over = oversample.fit_resample(
    x_train,
    y_train
)

# ============================================================
# CEK DISTRIBUSI SETELAH
# ============================================================

print("\nDistribusi target setelah Oversampling:\n")

print(y_train_over.value_counts())

Distribusi target sebelum Oversampling:

target
0    56078
1    23922
Name: count, dtype: int64

Distribusi target setelah Oversampling:

target
1    56078
0    56078
Name: count, dtype: int64


In [39]:
# Mengecek data setelah oversampling
print(x_train_over.shape)

(112156, 28)


In [40]:
print(x_train_over.dtypes)

age                      int64
gender                  object
height                 float64
weight                 float64
bmi                    float64
waist_size             float64
blood_pressure         float64
heart_rate             float64
cholesterol            float64
glucose                float64
insulin                float64
sleep_hours            float64
sleep_quality           object
work_hours             float64
physical_activity      float64
daily_steps            float64
calorie_intake         float64
sugar_intake           float64
alcohol_consumption     object
smoking_level           object
water_intake           float64
screen_time            float64
stress_level             int64
diet_type               object
exercise_type           object
device_usage            object
meals_per_day            int64
caffeine_intake         object
dtype: object


In [41]:
print(
    x_train_over.select_dtypes(
        include=['object','category']
    ).columns
)

Index(['gender', 'sleep_quality', 'alcohol_consumption', 'smoking_level', 'diet_type', 'exercise_type', 'device_usage', 'caffeine_intake'], dtype='object')


In [42]:
# ============================================================
# MODEL DASAR
# ============================================================

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=10,
    random_strength=5,
    bagging_temperature=1,
    loss_function='Logloss',
    eval_metric='F1',
    random_state=42,
    verbose=100,
    cat_features=cat_features
)

# ============================================================
# PARAMETER YANG DITUNING
# ============================================================

param_grid = {
    'depth': [5, 6],
    'learning_rate': [0.02, 0.03],
    'l2_leaf_reg': [10, 15],
    'random_strength': [5, 10],
    'bagging_temperature': [1, 3],
    'iterations': [1000]
}
# ============================================================
# GRID SEARCH
# ============================================================

grid_result = cat_model.grid_search(
    param_grid=param_grid,
    X=x_train_over,
    y=y_train_over,
    cv=3,
    partition_random_seed=42,
    verbose=True
)

# ============================================================
# PARAMETER TERBAIK
# ============================================================

print("Best Parameters:")
print(grid_result['params'])

0:	learn: 0.5620988	test: 0.5596737	best: 0.5596737 (0)	total: 213ms	remaining: 3m 32s
100:	learn: 0.5399076	test: 0.5206881	best: 0.5596737 (0)	total: 12.9s	remaining: 1m 54s
200:	learn: 0.5422477	test: 0.5196578	best: 0.5596737 (0)	total: 24.9s	remaining: 1m 39s
300:	learn: 0.5397453	test: 0.5156173	best: 0.5596737 (0)	total: 36.4s	remaining: 1m 24s
400:	learn: 0.5418178	test: 0.5166644	best: 0.5596737 (0)	total: 47.7s	remaining: 1m 11s
500:	learn: 0.5430517	test: 0.5173946	best: 0.5596737 (0)	total: 59.2s	remaining: 59s
600:	learn: 0.5490615	test: 0.5205247	best: 0.5596737 (0)	total: 1m 11s	remaining: 47.4s
700:	learn: 0.5717395	test: 0.5282968	best: 0.5596737 (0)	total: 1m 27s	remaining: 37.5s
800:	learn: 0.5955064	test: 0.5404229	best: 0.5596737 (0)	total: 1m 45s	remaining: 26.2s
900:	learn: 0.6102779	test: 0.5453175	best: 0.5596737 (0)	total: 2m 3s	remaining: 13.6s
999:	learn: 0.6226781	test: 0.5509109	best: 0.5596737 (0)	total: 2m 21s	remaining: 0us

bestTest = 0.5596737151
best

In [43]:
best_model = CatBoostClassifier(
    **grid_result['params'],
    loss_function='Logloss',
    eval_metric='F1',
    random_state=42,
    verbose=100,
    cat_features=cat_features
)

best_model.fit(
    x_train_over,
    y_train_over
)

0:	learn: 0.5626027	total: 171ms	remaining: 2m 50s
100:	learn: 0.5375732	total: 15.3s	remaining: 2m 15s
200:	learn: 0.5439852	total: 28.6s	remaining: 1m 53s
300:	learn: 0.5481991	total: 42.7s	remaining: 1m 39s
400:	learn: 0.5550801	total: 58.5s	remaining: 1m 27s
500:	learn: 0.5968225	total: 1m 19s	remaining: 1m 19s
600:	learn: 0.6327821	total: 1m 43s	remaining: 1m 8s
700:	learn: 0.6561435	total: 2m 6s	remaining: 53.9s
800:	learn: 0.6729402	total: 2m 29s	remaining: 37.2s
900:	learn: 0.6860879	total: 2m 53s	remaining: 19s
999:	learn: 0.6961180	total: 3m 14s	remaining: 0us


CatBoostClassifier(bagging_temperature=1, cat_features=['gender', 'sleep_quality', 'alcohol_consumption', 'smoking_level', 'diet_type', 'exercise_type', 'device_usage', 'caffeine_intake'], depth=6, eval_metric='F1', iterations=1000, l2_leaf_reg=10, learning_rate=0.03, loss_function='Logloss', random_state=42, random_strength=5, verbose=100)

Tes Data

In [44]:
# Prediksi langsung
y_pred = cat_model.predict(x_test)

# Evaluasi
print(classification_report(y_test, y_pred))

cat_acc = accuracy_score(y_test, y_pred)

print("Akurasi CatBoost = {:.2f}%".format(cat_acc*100))

              precision    recall  f1-score   support

           0       0.70      0.59      0.64     14019
           1       0.30      0.42      0.35      5981

    accuracy                           0.54     20000
   macro avg       0.50      0.50      0.50     20000
weighted avg       0.58      0.54      0.56     20000

Akurasi CatBoost = 53.92%
